Before you submit this problem, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name below:

In [ ]:
NAME = ""

In [ ]:
import IPython
assert IPython.version_info[0] >= 3, "Your version of IPython is too old, please update it."

---

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product
from tqdm import tqdm
from scipy.signal import wiener

In [ ]:
def read_as_RGB_and_GRAYSCALE(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image_rgb, image_gray


def add_awgn_noise(image, noise_std):
    """
    Adds AWGN noise to image based on the specified noise std.

    Parameters:
        - image
        - noise_std: Standard deviation of the AWGN noise

    Returns:
        Image with added AWGN noise

    Read more about AWGN noise here: https://en.wikipedia.org/wiki/Additive_white_Gaussian_noise
    """

    mean = 0
    noise = np.random.normal(mean, noise_std, image.shape)
    image_noisy = np.clip(image + noise, 0, 255).astype(np.uint8)

    return image_noisy


def add_awgn_noise_with_target_psnr(image, target_psnr):
    """
    Adds AWGN noise to image to achieve a target PSNR.

    Parameters:
        - image
        - target_psnr: Target PSNR in dB

    Returns:
        Image with added AWGN noise to achieve the target PSNR

    Read more about PSNR here: https://en.wikipedia.org/wiki/Peak_signal-to-noise_ratio
    """

    max_I = np.max(image)

    target_psnr_linear = 10 ** (target_psnr / 10)
    noise_variance = (max_I**2) / target_psnr_linear
    image_noisy = add_awgn_noise(image, np.sqrt(noise_variance))

    return image_noisy


def calculate_psnr(image_original, image_noisy):
    """
    Add salt and pepper noise to an image.

    Parameters:
        - image_original: Original image.
        - image_noisy: Noisy image.

    Returns:
        Peak Signal-to-Noise Ratio (PSNR) between the original and noisy images.

    Read more about PSNR here: https://en.wikipedia.org/wiki/Peak_signal-to-noise_ratio
    """

    mse = np.mean((image_original - image_noisy) ** 2)
    if mse == 0:
        return float("inf")
    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr


def minmax_norm(image):
    """
    Normalizes the input image to the range [0, 1]

    Parameters:
        - image: Input image to be normalized

    Returns:
        Normalized image
    """
    return (image - image.min()) / (image.max() - image.min())

In [ ]:
lenna, lenna_gray = read_as_RGB_and_GRAYSCALE("lenna.png")

---
## Part A, Singular Value Decomposition (SVD) watermarking (5 points)

1. Convert each character to its ASCII value using `ord`.
2. Convert the ASCII value to a binary string using `bin`, removing the prefix since it will loook like '0b*****'
3. Pad the binary string to 8 bits.
4. Convert the padded binary string to a list of bits.
5. Append the list of bits of each character to the output array.

In [ ]:
def string_to_bit_array(message):
    """
    Convert a string message to a bit array.

    Parameters:
        - message (str): The message to convert to a bit array.

    Returns:
        - bit_array (NumPy array): An array of bits representing the input message.

    Example:
        "b" -> [0, 1, 1, 0, 0, 0, 0, 1]
        "abc" -> [0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1]
    """

    # YOUR CODE HERE
    raise NotImplementedError()
    return np.asarray(bit_array, dtype=np.int8)

Obviously such function is invertable (meaning that we can convert the list of bits back to the string), but such function cannot deal with certain errors caused by incorrect watermark message decoding, which could hinder accurate conversion back to string.

In [ ]:
print(string_to_bit_array("abc"))

1. Convert the message to a binary sequence.
2. Decompose the image using SVD.
3. For each bit in the binary message, adjust the corresponding singular value by adding or subtracting the watermark strength.
4. Reconstruct the image from the modified singular values and the original U and V matrices.

In [ ]:
def embed_watermark_SVD(img, message_str, wm_strength=0.05):
    """
    Embeds a sequence of bits into the image using Singular Value Decomposition.

    Parameters:
        - img (NumPy array): The input image for watermarking
        - message_str (str): The message to embed into the image.
        - wm_strength (float): Strenth of the watermark, should be 0 < wm_strength < 1.

    Returns:
        - watermarked_img (NumPy array): The watermarked image.
        - info (dict): Dictionary with the necessary info for extracting of encoded message.

    Read more: https://en.wikipedia.org/wiki/Singular_value_decomposition
    """

    # YOUR CODE HERE
    raise NotImplementedError()
    info = {"message_length": len(message), "wm_strength": wm_strength}

    return watermarked_img, info

In [ ]:
lenna_wm, info = embed_watermark_SVD(lenna_gray, "lenna", wm_strength=0.1)
print(info)

fig, axs = plt.subplots(1, 3, figsize=(24, 8))
axs[0].imshow(lenna_gray, cmap="gray")
axs[0].set_title("Original image")
axs[1].imshow(lenna_wm, cmap="gray")
axs[1].set_title("Watermarked image")
axs[2].imshow(lenna_gray.astype(np.float32) - lenna_wm.astype(np.float32), cmap="gray")
axs[2].set_title("Difference between watermarked and original images")
plt.show()

1. Perform SVD on both the watermarked and original images to obtain their singular values.
2. Initialize an empty list for the message bits.
3. Iterate over the range of the message length specified in `info`.
4. For each index, calculate the difference between the corresponding singular values of the watermarked and original images.
5. Append `1` to the message if the difference is positive, otherwise append `0`.

In [ ]:
def extract_watermark_SVD(watermarked_img, original_img, info):
    """
    Extracts the watermark from a watermarked image using the original image.

    Parameters:
        - watermarked_img (NumPy array): The watermarked image. (1st output of embed_watermark_SVD)
        - original_img (NumPy array): The original image used for watermarking.
        - info (dict): Dictionary with the necessary info for extracting of encoded message. (2nd output of embed_watermark_SVD)

    Returns:
        - message_decoded (NumPy array): A sequence of bits decoded from watermarked image.

    """

    # YOUR CODE HERE
    raise NotImplementedError()

    return message_decoded

If you correctly implemented the functions for SVD watermarking you can try to play with this simulation which demonstrates the robustness to AWGN attack.

In [ ]:
def simulate_attack(
    imgs, messages_str, iterations=10, psnr_start=40, psnr_end=5, wm_strength=0.05
):
    for img, message_str in tqdm(list(product(imgs, messages_str))):
        results = []
        for iter, psnr in product(range(iterations), range(psnr_start, psnr_end, -1)):
            img_wm, info = embed_watermark_SVD(
                img, message_str, wm_strength=wm_strength
            )
            psnr_wm = calculate_psnr(img, img_wm)

            img_wm_noisy = add_awgn_noise_with_target_psnr(img_wm, psnr)
            message_decoded = extract_watermark_SVD(img_wm_noisy, img, info)
            ber = np.mean(
                np.bitwise_xor(string_to_bit_array(message_str), message_decoded)
            )

            results.append(
                {"Iteration": iter, "PSNR": psnr, "BER": ber, "PSNR_WM": psnr_wm}
            )

        dfp = (
            pd.DataFrame(results)
            .pivot_table(index="PSNR", values=["BER", "PSNR_WM"], aggfunc="mean")
            .reset_index()
        )
        plt.plot(
            dfp.PSNR,
            dfp.BER,
            label=f"BER of msg={message_str} image of size={img.shape}",
        )

    plt.title("Simulation results")
    plt.ylabel("Bit error rate (BER)")
    plt.xlabel("PSNR(img_wm, img_wm_noisy)")
    plt.legend(bbox_to_anchor=(1.9, 0.5))

Try to change the parameters of `simulate_attack_SVD` and `imgs & msgs` variables to see how does embedding parameter `wm_strength` impacts the decoding accuracy. Pay attention to the sizes of `imgs` and `msgs` and theirs impact as well.

In [ ]:
imgs = [cv2.resize(lenna_gray, (128, 128))]
msgs = ["a", "abc", "abcdf"]
simulate_attack(imgs, msgs, iterations=1, psnr_start=40, psnr_end=20, wm_strength=0.1)

In the cell below please give your explanation of the observed behaviour of SVD watermarking in a scope of the following aspects:
- What are the key advantages of such approach?
- To what attacks (besides of AWGN) is SVD watemarking robust?
- And to what attacks it is fragile?
- How do image and message sizes impact its performance?
- Is it possible to find an optimal combination of watermark strength & image size & message size? If yes - how?
- Do you have any ideas how to improve this approach by addressing its disadvantages?

YOUR ANSWER HERE

---
## Part B, Spatial Domain (SD) watermarking (5 points)

1. Calculate the total number of bits in the input bit array.
2. Determine the number of rows needed for the 2D array to square using the square root.
3. Calculate the total size of the 2D array by squaring the number of rows. If this size is less than the total number of bits, adjust by using an additional column.
4. If the calculated size is greater than the total number of bits, calculate the number of zeros needed to pad the array to the desired size.
5. Append the necessary number of zeros to the bit array to achieve the correct size for padding, if required.
6. Reshape the NumPy array into the 2D array with the calculated number of rows and an appropriate number of columns.

In [ ]:
def bit_array_to_2d_numpy(bit_array):
    """
    Converts a 1D bit array to a 2D numpy array with dimensions as square as possible.

    Parameters:
        - bit_array (list or NumPy array): The 1D array of bits to be converted into 2D.

    Returns:
        - bit_array_2d (NumPy array): The 2D numpy array resulting from the reshaping process.
        - N_pad (int): The number of zero bits added to the end of the bit_array to achieve a square 2D array.

    Example usage:
        bit_array = [1, 0, 1, 0, 1, 0]
        bit_array_2d, N_pad = bit_array_to_2d_numpy(bit_array)
    """

    # YOUR CODE HERE
    raise NotImplementedError()

    return bit_array_2d, N_pad

In [ ]:
message = string_to_bit_array('abcdef')
print(bit_array_to_2d_numpy(message))

1. Copy the original image to preserve the input.
2. Initialize an empty list to record the locations where patches will be inserted.
3. Determine the dimensions of the patch and the original image.
4. Loop through the desired number of patches to insert.
   - Randomly generate the top-left corner coordinates for the patch placement within the image bounds.
   - Check if the selected location overlaps with previously placed patches.
     - If there's overlap, skip the current iteration and attempt a new location.
   - Apply the watermark by adding the weighted patch to the image at the selected location.
     - Multiply the patch by the watermark strength before adding it to ensure it blends appropriately with the original image.
   - Record the location of the successfully inserted patch.
   - Exit the loop once the desired number of patches have been inserted without overlap.
5. Return the watermarked image and the list of patch locations.

In [ ]:
def embed_watermark_spatial(image, patch, wm_strength, number_of_patches):
    """
    Watermarks an image by inserting a given patch into random locations.

    Parameters:
        - image (numpy array): The original image to be watermarked.
        - patch (numpy array): The patch to be used as a watermark.
        - wm_strength (float): The strength of the watermark.
        - number_of_patches (int): The number of times the patch should be inserted.

    Returns:
        - watermarked_img (numpy array): The watermarked image.
        - locations (list of tuples): The locations where patches were inserted.
    """
    assert (
        image.shape[0] >= patch.shape[0] and image.shape[1] >= patch.shape[1]
    ), "Patch is too large."

    # YOUR CODE HERE
    raise NotImplementedError()

    return watermarked_img, locations

In [ ]:
message = string_to_bit_array("abcde")
patch, N_pad = bit_array_to_2d_numpy(message)

img_wm, locations = embed_watermark_spatial(lenna_gray, patch, 10, 20)

assert len(locations) == 20
print(locations)

fig, axs = plt.subplots(1, 3, figsize=(24, 8))
axs[0].imshow(lenna_gray, cmap="gray")
axs[0].set_title("Original image")
axs[1].imshow(img_wm, cmap="gray")
axs[1].set_title("Watermarked image")
axs[2].imshow(lenna_gray.astype(np.float32) - img_wm.astype(np.float32), cmap="gray")
axs[2].set_title("Difference between watermarked and original images")
plt.show()

1. Apply a Wiener `scipy.signal.wiener` filter to the watermarked image to estimate the watermark noise, calculating the difference between the filtered image and the original watermarked image.
3. Iterate over each location provided in the `locations` list:
4. For each location, extract a patch from the estimated watermark noise using the provided `x, y` coordinates and `patch_shape`.
5. Convert the list of patches into a NumPy array, organizing it such that each patch is a separate element in the first dimension.
6. Average the patches across the first dimension to combine them into a single patch representation.
7. Quantize the averaged patch by applying a threshold: convert values less than 0 to `True` (or 1) and values equal to or greater than 0 to `False` (or 0), creating a binary representation of the watermark.
8. Flatten the quantized array to create a 1D array of bits representing the extracted watermark message.

In [ ]:
def extract_watermark_spatial(watermarked_img, locations, patch_shape):
    """
    Extracts patches from the watermarked image based on the given locations.

    Parameters:
        - watermarked_img (numpy array): The watermarked image.
        - locations (list of tuples): The locations of the inserted patches. Each tuple contains (x, y) coordinates.
        - patch_shape (tuple): The shape of the patches to extract, specified as (height, width).

    Returns:
        - message_decoded (NumPy array): The array of extracted bits.
    """

    # YOUR CODE HERE
    raise NotImplementedError()

    return message_decoded

In [ ]:
message_decoded = extract_watermark_spatial(img_wm, locations, patch.shape)[:-N_pad]
print((message == message_decoded).all())

If you correctly implemented the functions for SD watermarking you can try to play with this simulation which demonstrates the robustness to AWGN attack.

In [ ]:
def simulate_attack_spatial(
    imgs,
    messages_str,
    iterations=10,
    psnr_start=40,
    psnr_end=5,
    wm_strength=3,
    n_patches=20,
):
    for img, message_str in tqdm(list(product(imgs, messages_str))):
        results = []
        for iter, psnr in product(range(iterations), range(psnr_start, psnr_end, -1)):

            message = string_to_bit_array(message_str)
            patch, N_pad = bit_array_to_2d_numpy(message)

            img_wm, locations = embed_watermark_spatial(
                img, patch, wm_strength, n_patches
            )
            psnr_wm = calculate_psnr(img, img_wm)
            img_wm_noisy = add_awgn_noise_with_target_psnr(img_wm, psnr)

            message_decoded = extract_watermark_spatial(
                img_wm_noisy, locations, patch.shape
            )[:-N_pad]

            ber = np.mean(np.bitwise_xor(message, message_decoded))

            results.append(
                {"Iteration": iter, "PSNR": psnr, "BER": ber, "PSNR_WM": psnr_wm}
            )

        dfp = (
            pd.DataFrame(results)
            .pivot_table(index="PSNR", values=["BER", "PSNR_WM"], aggfunc="mean")
            .reset_index()
        )
        plt.plot(
            dfp.PSNR,
            dfp.BER,
            label=f"BER of msg={message_str} image of size={img.shape}",
        )

    plt.title("Simulation results")
    plt.ylabel("Bit error rate (BER)")
    plt.xlabel("PSNR(img_wm, img_wm_noisy)")
    plt.legend(bbox_to_anchor=(1.9, 0.5))

Try to change the parameters of `simulate_attack_spatial` and `imgs & msgs` variables to see how does embedding parameters `wm_strength` and `n_patches` impact the decoding accuracy.

In [ ]:
imgs = [
    cv2.resize(lenna_gray, (128, 128)),
]
msgs = ["a", "abc", "abcdf"]
simulate_attack_spatial(
    imgs, msgs, iterations=1, psnr_start=40, psnr_end=20, wm_strength=10, n_patches=50
)

In the cell below please give your explanation of the observed behaviour of SD watermarking in a scope of the following aspects:
- What are the key advantages of such approach?
- To what attacks (besides of AWGN) is SD watemarking robust?
- And to what attacks it is fragile?
- How does image and message size impact the performance?
- How to fix the inconsistency in SD watermarking?
- Is it possible to find an optimal combination of watermark strength & number of patches? If yes - how?
- Is selecting random locations for embedding a message a good idea?
- Do you have any ideas how to improve this approach by addressing its disadvantages?

YOUR ANSWER HERE